In [24]:
!pip install -q catboost openpyxl

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import matplotlib.pyplot as plt

In [25]:
file_path = "/content/drive/MyDrive/supreme_court_judgments_enhanced.xlsx"

df = pd.read_excel(file_path)

print("Dataset shape:", df.shape)

Dataset shape: (26688, 41)


In [26]:
target = "estimated_duration_days"

# Remove target and identifier
drop_columns = [target]

if "filename" in df.columns:
    drop_columns.append("filename")

X = df.drop(columns=drop_columns).copy()
y = df[target].copy()

# Identify categorical columns
categorical_features = X.select_dtypes(
    include=["object", "category"]
).columns.tolist()

# Fill categorical missing values
for col in categorical_features:
    X[col] = X[col].fillna("unknown").astype(str)

print("Features:", X.shape[1])
print("Categorical features:", categorical_features)

Features: 39
Categorical features: ['case_number', 'citation', 'judgment_date', 'judges', 'bench', 'headnotes', 'acts_referred', 'petitioner', 'respondent', 'case_type', 'case_subtype', 'charges', 'sections_applied', 'penalties', 'imprisonment', 'fines', 'appellant', 'appeal_level', 'citation_pattern']


In [27]:
constant_columns = [
    col for col in X.columns
    if X[col].nunique(dropna=False) <= 1
]

X = X.drop(columns=constant_columns)

categorical_features = [
    col for col in categorical_features
    if col in X.columns
]

print("Remaining features:", X.shape[1])

Remaining features: 36


In [28]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=X["case_type"]
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=X_temp["case_type"]
)

print("Training:", len(X_train))
print("Validation:", len(X_val))
print("Testing:", len(X_test))

Training: 18681
Validation: 4003
Testing: 4004


In [29]:
cat_features = [
    X_train.columns.get_loc(col)
    for col in categorical_features
]

model = CatBoostRegressor(
    iterations=1500,
    learning_rate=0.03,
    depth=7,
    l2_leaf_reg=5,
    loss_function="RMSE",
    eval_metric="RMSE",
    random_seed=42,
    early_stopping_rounds=100,
    verbose=100,
    allow_writing_files=False
)

model.fit(
    X_train,
    y_train,
    cat_features=cat_features,
    eval_set=(X_val, y_val),
    use_best_model=True
)

0:	learn: 23.5647224	test: 23.4192701	best: 23.4192701 (0)	total: 274ms	remaining: 6m 50s
100:	learn: 8.4445280	test: 7.7556122	best: 7.7556122 (100)	total: 25.8s	remaining: 5m 56s
200:	learn: 8.1002559	test: 7.5257804	best: 7.5257804 (200)	total: 40.1s	remaining: 4m 19s
300:	learn: 8.0084714	test: 7.4996130	best: 7.4994500 (292)	total: 52.5s	remaining: 3m 28s
400:	learn: 7.9495258	test: 7.4867543	best: 7.4864308 (391)	total: 1m 3s	remaining: 2m 55s
500:	learn: 7.9032420	test: 7.4801575	best: 7.4788219 (444)	total: 1m 19s	remaining: 2m 38s
600:	learn: 7.8468040	test: 7.4739889	best: 7.4739334 (595)	total: 1m 34s	remaining: 2m 21s
700:	learn: 7.7682891	test: 7.4559066	best: 7.4559066 (700)	total: 1m 51s	remaining: 2m 6s
800:	learn: 7.6986546	test: 7.4443812	best: 7.4441802 (792)	total: 2m 9s	remaining: 1m 52s
900:	learn: 7.6256654	test: 7.4432148	best: 7.4422448 (884)	total: 2m 26s	remaining: 1m 37s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 7.442244798
bestItera

CatBoostRegressor(allow_writing_files=False, depth=7, early_stopping_rounds=100, eval_metric='RMSE', iterations=1500, l2_leaf_reg=5, learning_rate=0.03, loss_function='RMSE', random_seed=42, verbose=100)

In [30]:
predictions = model.predict(X_test)

In [31]:
tolerance = 0.10

accuracy = (
    np.abs(predictions - y_test.values) <=
    tolerance * y_test.values
).mean() * 100

print(f"Prediction Accuracy (±10%): {accuracy:.2f}%")

Prediction Accuracy (±10%): 98.25%


In [32]:
mae = mean_absolute_error(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))
r2 = r2_score(y_test, predictions)

print("\n===== CATBOOST RESULTS =====")
print(f"Accuracy ±10% : {accuracy:.2f}%")
print(f"MAE            : {mae:.2f} days")
print(f"RMSE           : {rmse:.2f} days")
print(f"R²             : {r2:.4f}")


===== CATBOOST RESULTS =====
Accuracy ±10% : 98.25%
MAE            : 2.06 days
RMSE           : 5.95 days
R²             : 0.9371


In [34]:
model.save_model(
    "/content/drive/MyDrive/court_duration_catboost_model.cbm"
)

print("\nModel saved successfully.")


Model saved successfully.


In [41]:
import pandas as pd

new_case = pd.DataFrame([{
    "year": 2026,
    "page_count": 35,
    "word_count": 15750,
    "character_count": 95000,
    "sentence_count": 520,
    "avg_word_per_page": 450,
    "case_number": "C-NEW-001",
    "citation": "2026 SCMR 100",
    "judgment_date": "2026-08-10",
    "judges": "Single Judge",
    "bench": "Single Bench",
    "headnotes": "Civil dispute regarding injunction",
    "acts_referred": "Code of Civil Procedure",
    "petitioner": "Party A",
    "respondent": "Party B",
    "case_type": "civil",
    "case_subtype": "injunction",
    "classification_confidence": 0.92,
    "complexity_score": 0.65,
    "case_age": 3,
    "charges": "none",
    "sections_applied": "Section 151 CPC",
    "penalties": "none",
    "imprisonment": "none",
    "fines": 0,
    "appellant": "none",
    "number_of_parties": 4,
    "total_citations": 18,
    "cases_cited_count": 15,
    "statutes_cited_count": 12,
    "foreign_citations": 2,
    "is_appeal": 0,
    "appeal_level": "none",
    "has_multiple_parties": 1,
    "has_headnotes": 1,
    "citation_pattern": "SCMR"
}])

In [44]:
prediction = catboost_model.predict(new_case)[0]
print("--------------------------------------")
print(f"Predicted Days   : {prediction:.2f}")
print(f"Predicted Months : {prediction / 30:.2f}")
print("======================================")

--------------------------------------
Predicted Days   : 125.16
Predicted Months : 4.17


In [56]:
# ============================================================
# TEST SAVED CATBOOST MODEL ON A REAL TEST CASE
# Handles missing categorical values
# ============================================================

import pandas as pd
from catboost import CatBoostRegressor
from sklearn.model_selection import train_test_split

# ------------------------------------------------------------
# 1. Load dataset
# ------------------------------------------------------------
file_path = "/content/drive/MyDrive/supreme_court_judgments_enhanced.xlsx"

df = pd.read_excel(file_path)

# ------------------------------------------------------------
# 2. Load saved model
# ------------------------------------------------------------
catboost_model = CatBoostRegressor()

catboost_model.load_model(
    "/content/drive/MyDrive/court_duration_catboost_model.cbm"
)

print("✅ Dataset loaded:", df.shape)
print("✅ CatBoost model loaded")


# ------------------------------------------------------------
# 3. Exact model features
# ------------------------------------------------------------
features = [
    "year",
    "page_count",
    "word_count",
    "character_count",
    "sentence_count",
    "avg_word_per_page",
    "case_number",
    "citation",
    "judgment_date",
    "judges",
    "bench",
    "headnotes",
    "acts_referred",
    "petitioner",
    "respondent",
    "case_type",
    "case_subtype",
    "classification_confidence",
    "complexity_score",
    "case_age",
    "charges",
    "sections_applied",
    "penalties",
    "imprisonment",
    "fines",
    "appellant",
    "number_of_parties",
    "total_citations",
    "cases_cited_count",
    "statutes_cited_count",
    "foreign_citations",
    "is_appeal",
    "appeal_level",
    "has_multiple_parties",
    "has_headnotes",
    "citation_pattern"
]


# ------------------------------------------------------------
# 4. Create same test split
# ------------------------------------------------------------
_, test_df = train_test_split(
    df,
    test_size=0.15,
    random_state=42,
    stratify=df["case_type"]
)

print("Test cases:", len(test_df))


# ------------------------------------------------------------
# 5. Select one real test case
# ------------------------------------------------------------
case = test_df.iloc[[0]].copy()


# ------------------------------------------------------------
# 6. Prepare features
# ------------------------------------------------------------
X_case = case[features].copy()


# ------------------------------------------------------------
# 7. Identify categorical features automatically
# ------------------------------------------------------------
categorical_features = [
    "case_number",
    "citation",
    "judgment_date",
    "judges",
    "bench",
    "headnotes",
    "acts_referred",
    "petitioner",
    "respondent",
    "case_type",
    "case_subtype",
    "charges",
    "sections_applied",
    "penalties",
    "imprisonment",
    "fines",
    "appellant",
    "appeal_level",
    "citation_pattern"
]


# ------------------------------------------------------------
# 8. Replace NaN in categorical features
# ------------------------------------------------------------
for col in categorical_features:
    if col in X_case.columns:
        X_case[col] = X_case[col].fillna("missing").astype(str)


# ------------------------------------------------------------
# 9. Prediction
# ------------------------------------------------------------
predicted = catboost_model.predict(X_case)[0]

actual = case["estimated_duration_days"].iloc[0]

difference = abs(predicted - actual)

within_10 = difference <= actual * 0.10

print(f"Actual Duration : {actual:.2f} days")
print(f"Predicted       : {predicted:.2f} days")
print(f"Difference      : {difference:.2f} days")

print("-" * 60)

if within_10:
    print("✅ Prediction is within ±10%")
else:
    print("❌ Prediction is outside ±10%")

print("=" * 60)

✅ Dataset loaded: (26688, 41)
✅ CatBoost model loaded
Test cases: 4004
Actual Duration : 120.00 days
Predicted       : 131.97 days
Difference      : 11.97 days
------------------------------------------------------------
✅ Prediction is within ±10%


In [57]:
model.save_model(
    "/content/drive/MyDrive/court_duration_catboost_model.cbm"
)

print("\nModel saved successfully.")


Model saved successfully.
